# ERAPave ML — Initial Guess Predictor

This notebook uses the **18 manually converged ERAPave rows (0–17)** in
`results_log_50.csv` as the supervised labels and the representative-location
Excel file as the source of pavement/TSD features.

### ML task

**Inputs → outputs**

`H1, H2, H3, BELLS temperature, D0, SCI300, selected road features`

→ `E1, E2, E3, E4`

The model is intended to provide a **good initial ERAPave guess**. ERAPave remains
the final back-calculation step.

### Important

Only rows with a converged solution in `results_log_50.csv` are used as training
labels. Do not use `d0_calc`, `sci300_calc`, `error_pct`, or `status` as predictors,
because those are outputs of the calibration process.


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# ---------------------------------------------------------
# FILES
# ---------------------------------------------------------
csv_file = Path(r"../data/results_log_50.csv")
excel_file = Path(r"../data/ERAPave_Training_50_Representative_Locations.xlsx")

solved = pd.read_csv(csv_file)
df = pd.read_excel(excel_file)

print("Solved CSV shape:", solved.shape)
print("Excel shape:", df.shape)
print("\nSolved columns:")
print(solved.columns.tolist())


Solved CSV shape: (18, 15)
Excel shape: (50, 60)

Solved columns:
['row_index', 'h1_cm', 'h2_cm', 'h3_cm', 'bells_temp', 'd0_target', 'sci300_target', 'E1_MPa', 'E2_MPa', 'E3_MPa', 'E4_MPa', 'd0_calc', 'sci300_calc', 'error_pct', 'status']


In [3]:
# ---------------------------------------------------------
# CHECK THE SOLVED LABEL DATA
# ---------------------------------------------------------

required_labels = [
    "row_index",
    "h1_cm", "h2_cm", "h3_cm", "bells_temp",
    "d0_target", "sci300_target",
    "E1_MPa", "E2_MPa", "E3_MPa", "E4_MPa"
]

missing = [c for c in required_labels if c not in solved.columns]
if missing:
    raise ValueError(f"Missing columns in results_log_50.csv: {missing}")

# Keep only converged solutions
solved = solved[solved["status"].astype(str).str.upper().eq("CONVERGED")].copy()

print("Converged training rows:", len(solved))
print("\nLabels:")
display(solved[
    ["row_index", "E1_MPa", "E2_MPa", "E3_MPa", "E4_MPa"]
])


Converged training rows: 18

Labels:


,row_index,E1_MPa,E2_MPa,E3_MPa,E4_MPa
0,0,1840,270,165,16.0
1,1,2330,250,130,37.0
2,2,1200,250,130,20.0
3,3,900,250,130,10.0
4,4,880,250,130,16.5
5,5,970,250,130,25.5
6,6,2200,250,130,42.0
7,7,3800,350,180,20.0
8,8,1400,130,100,20.0
9,9,870,400,128,20.0


In [4]:
# ---------------------------------------------------------
# CREATE ZERO-BASED ROW INDEX IN EXCEL
# ---------------------------------------------------------

df = df.copy()
df["row_index"] = np.arange(len(df))

# Check that all solved rows can be found in Excel
missing_rows = sorted(set(solved["row_index"]) - set(df["row_index"]))
if missing_rows:
    raise ValueError(f"These solved row_index values are missing from Excel: {missing_rows}")

# Merge Excel features with the solved labels
data = df.merge(
    solved[[
        "row_index",
        "h1_cm", "h2_cm", "h3_cm", "bells_temp",
        "d0_target", "sci300_target",
        "E1_MPa", "E2_MPa", "E3_MPa", "E4_MPa"
    ]],
    on="row_index",
    how="inner",
    suffixes=("_excel", "_csv")
)

print("Merged training data:", data.shape)


Merged training data: (18, 71)


## Feature sets

We use the CSV values for the six core calibration features because they are
already aligned with the solved ERAPave labels.

Additional Excel variables are tested separately. With only 18 training rows,
the expanded feature set is deliberately kept small.


In [5]:
# ---------------------------------------------------------
# CORE FEATURES FROM THE SOLVED CSV
# ---------------------------------------------------------

core_features = [
    "h1_cm",
    "h2_cm",
    "h3_cm",
    "bells_temp",
    "d0_target",
    "sci300_target"
]

# Candidate additional Excel features.
candidate_extra = [
    "Medelförtsd_1_SPEED",
    "Medelförpmsv4_AADT",
    "Medelförpmsv4_TAADT",
    "Distans",
    "Medelförghum_max",
    "Förstaförsoil_JG2",
    "Förstaförsoil_GJ2_tx",
    "Medelförtsd_1_ACD_PERCENT_WP_CELLS_CRACKED",
    "Medelförtsd_1_PCIP_14_DAYS",
    "Medelförmst_ditch_depth_10"
]

available_extra = [c for c in candidate_extra if c in df.columns]

print("Core features:")
print(core_features)

print("\nAdditional Excel features found:")
print(available_extra)


Core features:
['h1_cm', 'h2_cm', 'h3_cm', 'bells_temp', 'd0_target', 'sci300_target']

Additional Excel features found:
['Medelförtsd_1_SPEED', 'Medelförpmsv4_AADT', 'Medelförpmsv4_TAADT', 'Distans', 'Medelförghum_max', 'Förstaförsoil_JG2', 'Förstaförsoil_GJ2_tx', 'Medelförtsd_1_ACD_PERCENT_WP_CELLS_CRACKED', 'Medelförtsd_1_PCIP_14_DAYS', 'Medelförmst_ditch_depth_10']


In [6]:
# ---------------------------------------------------------
# PREPARE CORE AND EXPANDED DATASETS
# ---------------------------------------------------------

targets = ["E1_MPa", "E2_MPa", "E3_MPa", "E4_MPa"]

X_core = solved[core_features].copy()

# Add extra Excel variables using row_index
extra_lookup = df[["row_index"] + available_extra].copy()
training = solved[[
    "row_index",
    "E1_MPa", "E2_MPa", "E3_MPa", "E4_MPa"
] + core_features].copy()

training = training.merge(
    extra_lookup,
    on="row_index",
    how="left"
)

X_expanded = training[core_features + available_extra].copy()
y = training[targets].copy()

def clean_numeric(X):
    X = X.copy()
    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors="coerce")
    # Drop unusable columns
    usable = [
        c for c in X.columns
        if X[c].notna().sum() >= 3 and X[c].nunique(dropna=True) > 1
    ]
    X = X[usable]
    return X

X_core = clean_numeric(X_core)
X_expanded = clean_numeric(X_expanded)

print("Core feature count:", X_core.shape[1])
print("Expanded feature count:", X_expanded.shape[1])
print("\nExpanded features:")
print(X_expanded.columns.tolist())


Core feature count: 6
Expanded feature count: 15

Expanded features:
['h1_cm', 'h2_cm', 'h3_cm', 'bells_temp', 'd0_target', 'sci300_target', 'Medelförtsd_1_SPEED', 'Medelförpmsv4_AADT', 'Medelförpmsv4_TAADT', 'Distans', 'Medelförghum_max', 'Förstaförsoil_JG2', 'Medelförtsd_1_ACD_PERCENT_WP_CELLS_CRACKED', 'Medelförtsd_1_PCIP_14_DAYS', 'Medelförmst_ditch_depth_10']


## Model selection

Because there are only 18 solved rows, this notebook does **not** use a neural
network.

We compare:
- Random Forest
- Extra Trees
- Gradient Boosting

Each modulus is predicted separately. Leave-one-out cross-validation is used
because the training set is very small.


In [7]:
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=4,
        min_samples_leaf=2,
        random_state=42
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=300,
        max_depth=4,
        min_samples_leaf=2,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        max_depth=2,
        learning_rate=0.03,
        loss="huber",
        random_state=42
    )
}

loo = LeaveOneOut()

def evaluate_feature_set(X, name):
    rows = []

    for target in targets:
        for model_name, model in models.items():
            pipe = make_pipeline(
                SimpleImputer(strategy="median"),
                model
            )

            scores = cross_val_score(
                pipe,
                X,
                y[target],
                cv=loo,
                scoring="neg_mean_absolute_error"
            )

            rows.append({
                "feature_set": name,
                "target": target,
                "model": model_name,
                "LOO_MAE_MPa": -scores.mean()
            })

    return pd.DataFrame(rows)

results_core = evaluate_feature_set(X_core, "Core")
results_expanded = evaluate_feature_set(X_expanded, "Expanded")

results = pd.concat([results_core, results_expanded], ignore_index=True)

display(
    results.sort_values(
        ["target", "LOO_MAE_MPa"]
    )
)


,feature_set,target,model,LOO_MAE_MPa
13,Expanded,E1_MPa,Extra Trees,467.955007
12,Expanded,E1_MPa,Random Forest,472.714862
0,Core,E1_MPa,Random Forest,473.808065
1,Core,E1_MPa,Extra Trees,498.847348
2,Core,E1_MPa,Gradient Boosting,515.280626
14,Expanded,E1_MPa,Gradient Boosting,539.481897
3,Core,E2_MPa,Random Forest,82.326015
4,Core,E2_MPa,Extra Trees,84.161697
16,Expanded,E2_MPa,Extra Trees,84.234297
15,Expanded,E2_MPa,Random Forest,86.078323


In [8]:
# ---------------------------------------------------------
# SELECT THE BEST FEATURE SET + MODEL FOR EACH TARGET
# ---------------------------------------------------------

best_rows = (
    results
    .sort_values(["target", "LOO_MAE_MPa"])
    .groupby("target", as_index=False)
    .first()
)

print("Best combination for each modulus:")
display(best_rows)


Best combination for each modulus:


,target,feature_set,model,LOO_MAE_MPa
0,E1_MPa,Expanded,Extra Trees,467.955007
1,E2_MPa,Core,Random Forest,82.326015
2,E3_MPa,Core,Random Forest,21.164663
3,E4_MPa,Core,Extra Trees,5.697037


In [9]:
# ---------------------------------------------------------
# TRAIN FINAL MODELS USING THE BEST COMBINATION
# ---------------------------------------------------------

X_sets = {
    "Core": X_core,
    "Expanded": X_expanded
}

trained_models = {}
selected_features = {}

for _, r in best_rows.iterrows():
    target = r["target"]
    feature_set = r["feature_set"]
    model_name = r["model"]

    X_train = X_sets[feature_set]
    model = models[model_name]

    pipe = make_pipeline(
        SimpleImputer(strategy="median"),
        model
    )

    pipe.fit(X_train, y[target])

    trained_models[target] = pipe
    selected_features[target] = X_train.columns.tolist()

    print(
        f"{target}: {feature_set} + {model_name}"
    )


E1_MPa: Expanded + Extra Trees
E2_MPa: Core + Random Forest
E3_MPa: Core + Random Forest
E4_MPa: Core + Extra Trees


## Predict Row 

The model uses the features available for Row 18 in the Excel file plus the
target D0/SCI300 from the solved/measurement data.

Set `next_row_index` to 19, 20, etc. when continuing.


In [34]:
import pandas as pd

# ============================================
# LOAD DATASET
# ============================================

df = pd.read_excel(
    '../data/ERAPave_Training_50_Representative_Locations.xlsx'
)

# ============================================
# SELECT ROW
# ============================================
next_row_index = 23

row = df.iloc[next_row_index]

# ============================================
# CREATE ML INPUT
# ============================================

ml_row = pd.DataFrame([{
    "h1_cm": float(row["Medelförmst_Layer_1_thk"]) * 100,
    "h2_cm": float(row["Medelförmst_Layer_2_thk"]) * 100,
    "h3_cm": float(row["Medelförmst_Layer_3_thk"]) * 100,
    "bells_temp": float(row["Medelförtsd_1_BELLS_TEMP"]),
    "d0_target": abs(float(row["Medelförtsd_1_D0000"])),
    "sci300_target": float(row["Medelförtsd_1_SCI_300"])
}])

X_next = ml_row[core_features]

print(f"Row {next_row_index} input:")
print(ml_row)

print("\nInput shape:", X_next.shape)


# ============================================
# PREDICT FOUR MODULI
# ============================================

prediction = {}

for target in targets:
    prediction[target] = float(
        trained_models[target].predict(X_next)[0]
    )

print("\nRaw ML prediction:")

for target, value in prediction.items():
    print(f"{target}: {value:.2f} MPa")


# ============================================
# PRACTICAL ERAPAVE VALUES
# ============================================

practical_prediction = {
    "E1_MPa": round(prediction["E1_MPa"] / 10) * 10,
    "E2_MPa": round(prediction["E2_MPa"] / 5) * 5,
    "E3_MPa": round(prediction["E3_MPa"] / 5) * 5,
    "E4_MPa": round(prediction["E4_MPa"])
}

print("\nPractical ERAPave starting values:")

for target, value in practical_prediction.items():
    print(f"{target}: {value} MPa")


# ============================================
# THICKNESSES
# ============================================

h1 = float(row["Medelförmst_Layer_1_thk"]) * 100
h2 = float(row["Medelförmst_Layer_2_thk"]) * 100
h3 = float(row["Medelförmst_Layer_3_thk"]) * 100


# ============================================
# USE ML PREDICTION
# ============================================

E1 = practical_prediction["E1_MPa"]
E2 = practical_prediction["E2_MPa"]
E3 = practical_prediction["E3_MPa"]
E4 = practical_prediction["E4_MPa"]


# ============================================
# CREATE ERAPAVE DAT FILE
# ============================================

filename = f"row{next_row_index}.dat"

with open(filename, "w") as f:

    f.write("MATERIAL PROPERTIES\n")
    f.write("4\n")
    f.write("0\n")

    f.write(f"1 {h1:.4f} {E1:.2f} 0.35 24\n")
    f.write(f"2 {h2:.4f} {E2:.2f} 0.35 22\n")
    f.write(f"3 {h3:.4f} {E3:.2f} 0.35 20\n")
    f.write(f"4 1e+300 {E4:.2f} 0.40 18\n")

    f.write("\nLOADING\n")
    f.write("1\n")
    f.write("700.0\n")
    f.write("50.0\n")
    f.write("0\n")
    f.write("0\n")

    f.write("\nEVALUATION LOCATIONS\n")
    f.write("7\n")
    f.write("x y z\n")
    f.write("0 0 0\n")
    f.write("20 0 0\n")
    f.write("30 0 0\n")
    f.write("60 0 0\n")
    f.write("90 0 0\n")
    f.write("120 0 0\n")
    f.write("150 0 0\n")


# ============================================
# PRINT
# ============================================

print("\n============================================")
print("ERAPave input file created")
print("============================================")

print(f"Created: {filename}")
print(f"Layer 1 Thickness (cm): {h1:.4f}")
print(f"Layer 2 Thickness (cm): {h2:.4f}")
print(f"Layer 3 Thickness (cm): {h3:.4f}")

print(f"BELLS Temp (°C):        {row['Medelförtsd_1_BELLS_TEMP']:.2f}")
print(f"Target D0 (µm):         {abs(row['Medelförtsd_1_D0000']):.2f}")
print(f"Target SCI300 (µm):     {row['Medelförtsd_1_SCI_300']:.2f}")

print(f"Moduli (MPa):           {E1}, {E2}, {E3}, {E4}")

Row 23 input:
       h1_cm     h2_cm      h3_cm  bells_temp  d0_target  sci300_target
0  13.988682  37.08476  88.262911   20.752864    681.945         270.63

Input shape: (1, 6)

Raw ML prediction:
E1_MPa: 843.73 MPa
E2_MPa: 192.92 MPa
E3_MPa: 94.27 MPa
E4_MPa: 19.94 MPa

Practical ERAPave starting values:
E1_MPa: 840 MPa
E2_MPa: 195 MPa
E3_MPa: 95 MPa
E4_MPa: 20 MPa

ERAPave input file created
Created: row23.dat
Layer 1 Thickness (cm): 13.9887
Layer 2 Thickness (cm): 37.0848
Layer 3 Thickness (cm): 88.2629
BELLS Temp (°C):        20.75
Target D0 (µm):         681.95
Target SCI300 (µm):     270.63
Moduli (MPa):           840, 195, 95, 20


In [27]:
# ---------------------------------------------------------
# PRACTICAL ERAPAVE INITIAL GUESS
# ---------------------------------------------------------

def round_modulus(value, step):
    return int(round(value / step) * step)

initial_guess = {
    "E1_MPa": round_modulus(prediction["E1_MPa"], 10),
    "E2_MPa": round_modulus(prediction["E2_MPa"], 5),
    "E3_MPa": round_modulus(prediction["E3_MPa"], 5),
    "E4_MPa": round_modulus(prediction["E4_MPa"], 1)
}

print(f"\nRecommended ERAPave starting guess for Row {next_row_index}:")
print(
    f'{initial_guess["E1_MPa"]} / '
    f'{initial_guess["E2_MPa"]} / '
    f'{initial_guess["E3_MPa"]} / '
    f'{initial_guess["E4_MPa"]} MPa'
)



Recommended ERAPave starting guess for Row 19:
1200 / 260 / 130 / 22 MPa


In [41]:
"""
ERAPAVE-like Python forward model — Version 1

Purpose
-------
First validation step for the pavement back-calculation project.

This program uses the existing Burmister/MLET Python solver and applies
two calibration factors so that the Python response matches a supplied
ERAPAVE benchmark at:
    r = 0 cm  -> D0
    r = 30 cm -> D30

IMPORTANT
---------
This is NOT yet a full reproduction of ERAPAVE 2012. The ERAPAVE manual
describes additional numerical details such as selected-point stress
coefficients, piecewise-linear interpolation and Richardson extrapolation.
Those implementation details are not fully specified in the manual.

The calibration factors are therefore explicitly separated from the
physics solver. They should be re-estimated using multiple ERAPAVE
benchmarks before using this for the 15,000-row back-calculation.

Requirements:
    pip install numpy scipy
"""

import numpy as np
from scipy import special
import scipy.linalg as linalg

def erapave_mlet(
    q_kpa,
    radius_cm,
    r_cm,
    z_cm,
    thickness_cm,
    modulus_mpa,
    poisson,
    iteration=80,
    bounded=True,
):
    """
    Burmister multilayer elastic solver following the public PyMastic
    formulation, with ERAPave-style benchmark settings.

    Units:
      q_kpa       : kPa
      radius_cm   : cm
      r_cm, z_cm  : cm
      thickness_cm: cm
      modulus_mpa : MPa
      output      : displacement in micrometres

    This is a linear-elastic MLET forward solver. It is NOT claimed to
    reproduce proprietary ERAPave implementation details until validated
    against ERAPave output.
    """

    x = np.atleast_1d(np.asarray(r_cm, dtype=float)).copy()
    z = np.atleast_1d(np.asarray(z_cm, dtype=float)).copy()
    x[x == 0] = 1e-8
    z[z == 0] = 1e-8

    H = np.asarray(thickness_cm, dtype=float)
    E = np.asarray(modulus_mpa, dtype=float) * 1000.0  # kPa
    nu = np.asarray(poisson, dtype=float)

    n_layers = len(E)
    if len(H) != n_layers - 1:
        raise ValueError("For N layers, thickness_cm must contain N-1 finite-layer thicknesses.")
    if len(nu) != n_layers:
        raise ValueError("poisson must have one value per layer.")

    # Dimensionless normalization used by the PyMastic formulation.
    sumH = np.sum(H)
    lam = np.hstack((0.0, np.cumsum(H) / sumH, 1000.0))
    L = z / sumH
    ro = x / sumH
    alpha = radius_cm / sumH

    # Bessel zeros. We generate more than needed instead of hard-coding
    # the published tables.
    z0 = special.jn_zeros(0, max(iteration + 20, 120))
    z1 = special.jn_zeros(1, max(iteration + 20, 120))

    z0_scaled = (z0[None, :] / ro[:, None]).T
    z1_scaled = z1 / alpha

    bessel_zeros = np.hstack(
        (np.array([0.0]), z0_scaled.flatten(), z1_scaled)
    )
    bessel_zeros = np.sort(bessel_zeros)

    D1 = (bessel_zeros[1] - bessel_zeros[0]) / 6.0 - 1e-5
    D2 = (bessel_zeros[2] - bessel_zeros[1]) / 2.0 - 1e-5

    aux1 = np.arange(bessel_zeros[0], bessel_zeros[1], D1)
    aux2 = np.arange(bessel_zeros[1], bessel_zeros[2], D2)

    # Same quadrature construction as the public PyMastic formulation.
    m_values = np.hstack((aux1, aux2[1:], bessel_zeros[3:iteration]))

    dm = np.diff(m_values)

    m_matrix = np.vstack((m_values, m_values, m_values, m_values)).T

    coefficient = np.empty((len(dm), 4))
    coefficient[:, 0] = dm / 2 - 0.86114 * dm / 2
    coefficient[:, 1] = dm / 2 - 0.33998 * dm / 2
    coefficient[:, 2] = dm / 2 + 0.33998 * dm / 2
    coefficient[:, 3] = dm / 2 + 0.86114 * dm / 2

    ft_gauss = np.empty((4, len(dm)))
    ft_gauss[0, :] = 0.34786 * dm / 2
    ft_gauss[1, :] = 0.65215 * dm / 2
    ft_gauss[2, :] = 0.65215 * dm / 2
    ft_gauss[3, :] = 0.34786 * dm / 2

    ft = ft_gauss.flatten(order="F")

    m_final = m_matrix[:-1, :] + coefficient
    m = np.sort(m_final.flatten(order="F"))

    # Recreate the quadrature weights associated with sorted m.
    # The public implementation sorts m but keeps ft in its original
    # flattening order. We retain that behavior for compatibility.
    n_m = len(m)

    ind = np.zeros(len(z), dtype=int)
    for i in range(len(z)):
        ind[i] = np.where(lam > L[i])[0][0]

    A = np.zeros((n_m, n_layers))
    B = np.zeros((n_m, n_layers))
    C = np.zeros((n_m, n_layers))
    D = np.zeros((n_m, n_layers))

    H_BC = np.hstack((H, np.max(H) * 1000.0))
    lam_BC = np.cumsum(H_BC) / sumH

    R = E[:-1] / E[1:] * ((1.0 + nu[1:]) / (1.0 + nu[:-1]))

    for j in range(n_m):
        mj = m[j]

        left1 = np.array([
            [np.exp(-mj * lam_BC[0]), 1.0],
            [np.exp(-mj * lam_BC[0]), -1.0]
        ])

        right1 = np.array([
            [-(1 - 2 * nu[0]) * np.exp(-mj * lam_BC[0]), 1 - 2 * nu[0]],
            [2 * nu[0] * np.exp(-mj * lam_BC[0]), 2 * nu[0]]
        ])

        dlam = np.diff(np.hstack((0.0, lam_BC)))
        F = np.exp(-mj * dlam)

        solved = []

        for i in range(n_layers - 1):
            if bounded:
                left = np.array([
                    [1, F[i],
                     -(1 - 2*nu[i] - mj*lam_BC[i]),
                     (1 - 2*nu[i] + mj*lam_BC[i])*F[i]],

                    [1, -F[i],
                     2*nu[i] + mj*lam_BC[i],
                     (2*nu[i] - mj*lam_BC[i])*F[i]],

                    [1, F[i],
                     1 + mj*lam_BC[i],
                     -(1 - mj*lam_BC[i])*F[i]],

                    [1, -F[i],
                     -(2 - 4*nu[i] - mj*lam_BC[i]),
                     -(2 - 4*nu[i] + mj*lam_BC[i])*F[i]]
                ], dtype=float)

                right = np.array([
                    [F[i+1], 1,
                     -(1 - 2*nu[i+1] - mj*lam_BC[i])*F[i+1],
                     1 - 2*nu[i+1] + mj*lam_BC[i]],

                    [F[i+1], -1,
                     (2*nu[i+1] + mj*lam_BC[i])*F[i+1],
                     2*nu[i+1] - mj*lam_BC[i]],

                    [R[i]*F[i+1], R[i],
                     (1 + mj*lam_BC[i])*R[i]*F[i+1],
                     -(1 - mj*lam_BC[i])*R[i]],

                    [R[i]*F[i+1], -R[i],
                     -(2 - 4*nu[i+1] - mj*lam_BC[i])*R[i]*F[i+1],
                     -(2 - 4*nu[i+1] + mj*lam_BC[i])*R[i]]
                ], dtype=float)

            else:
                # Frictionless interface.
                left = np.array([
                    [1, F[i],
                     -(1 - 2*nu[i] - mj*lam_BC[i]),
                     (1 - 2*nu[i] + mj*lam_BC[i])*F[i]],

                    [1, -F[i],
                     -(2 - 4*nu[i] - mj*lam_BC[i]),
                     -(2 - 4*nu[i] + mj*lam_BC[i])*F[i]],

                    [1, -F[i],
                     2*nu[i] + mj*lam_BC[i],
                     (2*nu[i] - mj*lam_BC[i])*F[i]],

                    [1e-3, 1e-3, 1e-3, 1e-3]
                ], dtype=float)

                right = np.array([
                    [F[i+1], 1,
                     -(1 - 2*nu[i+1] - mj*lam_BC[i])*F[i+1],
                     1 - 2*nu[i+1] + mj*lam_BC[i]],

                    [R[i]*F[i+1], -R[i],
                     -(2 - 4*nu[i+1] - mj*lam_BC[i])*R[i]*F[i+1],
                     -(2 - 4*nu[i+1] + mj*lam_BC[i])*R[i]],

                    [1e-3, 1e-3, 1e-3, 1e-3],

                    [F[i+1], -1,
                     (2*nu[i+1] + mj*lam_BC[i])*F[i+1],
                     2*nu[i+1] - mj*lam_BC[i]]
                ], dtype=float)

            solved.append(np.linalg.solve(left, right))

        transfer = solved[0]
        for s in solved[1:]:
            transfer = transfer @ s

        BnDn_matrix = transfer[:, [1, 3]]

        M = np.hstack((left1, right1)) @ BnDn_matrix

        try:
            BnDn = np.linalg.solve(M, np.array([[1.0], [0.0]]))
        except np.linalg.LinAlgError:
            BnDn = np.linalg.pinv(M) @ np.array([[1.0], [0.0]])

        A_bc = np.zeros(n_layers)
        B_bc = np.zeros(n_layers)
        C_bc = np.zeros(n_layers)
        D_bc = np.zeros(n_layers)

        B_bc[-1] = BnDn[0, 0]
        D_bc[-1] = BnDn[1, 0]

        for i in reversed(range(n_layers - 1)):
            bc = solved[i] @ np.array([
                A_bc[i+1], B_bc[i+1], C_bc[i+1], D_bc[i+1]
            ])
            A_bc[i], B_bc[i], C_bc[i], D_bc[i] = bc

        A[j, :] = A_bc
        B[j, :] = B_bc
        C[j, :] = C_bc
        D[j, :] = D_bc

    # Surface vertical displacement only.
    displacement_cm = np.zeros(len(x))

    for j, r in enumerate(ro):
        for i in range(len(z)):
            layer = ind[i] - 1

            e = E[layer]
            v = nu[layer]

            upper = lam[ind[i]] - L[i]
            lower = L[i] - lam[ind[i]-1]

            rs = (
                -((1 + v) / e)
                * special.jv(0, m * r)
                * (
                    (A[:, layer] - C[:, layer] *
                     (2 - 4*v - m*L[i]))
                    * np.exp(-m * upper)
                    -
                    (B[:, layer] + D[:, layer] *
                     (2 - 4*v + m*L[i]))
                    * np.exp(-m * lower)
                )
            )

            displacement_cm[j] = (
                sumH * q_kpa * alpha
                * np.sum(ft * rs * special.jv(1, m * alpha) / m)
            )

    return displacement_cm * 10000.0


# ============================================================
# ERAPAVE BENCHMARK
# ============================================================


def contact_radius_cm(load_kn, pressure_kpa):
    return np.sqrt(load_kn / (np.pi * pressure_kpa)) * 100.0


def calibrate_to_erapave(raw_d0, raw_d30, erapave_d0, erapave_d30):
    """
    Determine two simple response calibration factors.

    center_factor is applied at r=0.
    far_field_factor is applied at r>0.

    This is a validation/calibration device, not an ERAPAVE equation.
    """
    far_field_factor = erapave_d30 / raw_d30
    center_factor = erapave_d0 / (raw_d0 * far_field_factor)

    return far_field_factor, center_factor


def erapave_like(
    thickness_cm,
    modulus_mpa,
    poisson,
    pressure_kpa=700.0,
    load_kn=50.0,
    locations_cm=None,
    far_field_factor=1.0,
    center_factor=1.0,
):
    if locations_cm is None:
        locations_cm = np.array([0, 20, 30, 60, 90, 120, 150], dtype=float)
    else:
        locations_cm = np.asarray(locations_cm, dtype=float)

    radius_cm = contact_radius_cm(load_kn, pressure_kpa)
    z_cm = np.zeros_like(locations_cm)

    raw = erapave_mlet(
        q_kpa=pressure_kpa,
        radius_cm=radius_cm,
        r_cm=locations_cm,
        z_cm=z_cm,
        thickness_cm=np.asarray(thickness_cm, dtype=float),
        modulus_mpa=np.asarray(modulus_mpa, dtype=float),
        poisson=np.asarray(poisson, dtype=float),
        iteration=80,
        bounded=True,
    )

    calibrated = raw * far_field_factor
    calibrated[np.isclose(locations_cm, 0.0)] *= center_factor

    d0 = calibrated[np.argmin(np.abs(locations_cm - 0.0))]
    d30 = calibrated[np.argmin(np.abs(locations_cm - 30.0))]
    sci300 = d0 - d30

    return locations_cm, raw, calibrated, d0, d30, sci300


if __name__ == "__main__":

    # ---------------------------------------------------------
    # YOUR CURRENT ROW 23 EXAMPLE
    # ---------------------------------------------------------
    thickness_cm = [13.989, 37.085, 88.263]
    modulus_mpa = [650.0, 250.0, 150.0, 20.0]
    poisson = [0.35, 0.35, 0.35, 0.40]

    pressure_kpa = 700.0
    load_kn = 50.0

    # ERAPAVE reference from the user's run:
    erapave_d0 = 685.3
    erapave_d30 = 351.9

    locations, raw, _, _, _, _ = erapave_like(
        thickness_cm,
        modulus_mpa,
        poisson,
        pressure_kpa,
        load_kn,
    )

    raw_d0 = raw[np.argmin(np.abs(locations - 0.0))]
    raw_d30 = raw[np.argmin(np.abs(locations - 30.0))]

    far_factor, center_factor = calibrate_to_erapave(
        raw_d0, raw_d30, erapave_d0, erapave_d30
    )

    locations, raw, calibrated, d0, d30, sci = erapave_like(
        thickness_cm,
        modulus_mpa,
        poisson,
        pressure_kpa,
        load_kn,
        far_field_factor=far_factor,
        center_factor=center_factor,
    )

    print("\n" + "=" * 70)
    print("PYTHON ERAPAVE-LIKE MODEL — FIRST BENCHMARK")
    print("=" * 70)
    print(f"E = {modulus_mpa} MPa")
    print(f"Thickness = {thickness_cm} cm")
    print(f"Pressure = {pressure_kpa} kPa")
    print(f"Load = {load_kn} kN")
    print(f"Contact radius = {contact_radius_cm(load_kn, pressure_kpa):.4f} cm")

    print("\nCalibration factors")
    print(f"far_field_factor = {far_factor:.6f}")
    print(f"center_factor    = {center_factor:.6f}")

    print("\nDisplacement comparison")
    print(f"Raw Python D0   = {raw_d0:.2f} um")
    print(f"ERAPAVE D0      = {erapave_d0:.2f} um")
    print(f"Python D0       = {d0:.2f} um")

    print(f"\nRaw Python D30  = {raw_d30:.2f} um")
    print(f"ERAPAVE D30     = {erapave_d30:.2f} um")
    print(f"Python D30      = {d30:.2f} um")

    print(f"\nPython SCI300   = {sci:.2f} um")
    print(f"ERAPAVE SCI300  = {erapave_d0 - erapave_d30:.2f} um")

    print("\nAll locations after calibration")
    print("r(cm)     raw(um)     calibrated(um)")
    for r, a, b in zip(locations, raw, calibrated):
        print(f"{r:5.0f}     {a:9.2f}     {b:14.2f}")

    print("\nNOTE:")
    print("The two calibration factors are fitted to this benchmark only.")
    print("Do not use them for 15,000-row prediction until validated")
    print("against several independent ERAPAVE runs.")



PYTHON ERAPAVE-LIKE MODEL — FIRST BENCHMARK
E = [650.0, 250.0, 150.0, 20.0] MPa
Thickness = [13.989, 37.085, 88.263] cm
Pressure = 700.0 kPa
Load = 50.0 kN
Contact radius = 15.0786 cm

Calibration factors
far_field_factor = 0.571536
center_factor    = 1.274669

Displacement comparison
Raw Python D0   = 940.68 um
ERAPAVE D0      = 685.30 um
Python D0       = 685.30 um

Raw Python D30  = 615.71 um
ERAPAVE D30     = 351.90 um
Python D30      = 351.90 um

Python SCI300   = 333.40 um
ERAPAVE SCI300  = 333.40 um

All locations after calibration
r(cm)     raw(um)     calibrated(um)
    0        940.68             685.30
   20        707.77             404.51
   30        615.71             351.90
   60        488.24             279.05
   90        428.43             244.87
  120        385.98             220.60
  150        351.94             201.15

NOTE:
The two calibration factors are fitted to this benchmark only.
Do not use them for 15,000-row prediction until validated
against several 

In [58]:
# ============================================================
# ERAPAVE-LIKE MODEL — VERSION 2.2
# JUPYTER NOTEBOOK VERSION
#
# PURPOSE
# -------
# Improve V2.1 by allowing additional ERAPAVE calibration
# points to be added safely.
#
# IMPORTANT
# ---------
# D0     = displacement at x = 0 cm
# D30    = displacement at x = 30 cm
# SCI300 = D0 - D30
#
# DO NOT enter guessed ERAPAVE values.
# Only add a calibration row after the ERAPAVE run has
# actually been performed.
#
# This remains an ERAPAVE-calibrated MLET approximation,
# NOT an exact reproduction of ERAPAVE 2012.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import importlib.util

import numpy as np
from scipy.optimize import least_squares


# ============================================================
# 2. LOAD RAW MLET SOLVER
# ============================================================

HERE = Path.cwd()

RAW_SOLVER = HERE / "ERAPave_MLET_V3_RAW_Benchmark.py"

if not RAW_SOLVER.exists():

    raise FileNotFoundError(
        "\nCould not find:\n"
        f"{RAW_SOLVER}\n\n"
        "Put ERAPave_MLET_V3_RAW_Benchmark.py "
        "in the same folder as this notebook."
    )


spec = importlib.util.spec_from_file_location(
    "raw_mlet",
    RAW_SOLVER
)

raw_mlet = importlib.util.module_from_spec(spec)

spec.loader.exec_module(raw_mlet)

print("Raw MLET solver loaded successfully.")


# ============================================================
# 3. ROW 23 STRUCTURE
# ============================================================

thickness_cm = np.array(
    [
        13.989,
        37.085,
        88.263
    ],
    dtype=float
)


poisson = np.array(
    [
        0.35,
        0.35,
        0.35,
        0.40
    ],
    dtype=float
)


pressure_kpa = 700.0

load_kn = 50.0


# ------------------------------------------------------------
# ERAPAVE output locations
#
# index 0 = x = 0 cm
# index 1 = x = 20 cm
# index 2 = x = 30 cm
# ------------------------------------------------------------

locations_cm = np.array(
    [
        0,
        20,
        30,
        60,
        90,
        120,
        150
    ],
    dtype=float
)


# Contact radius

contact_radius_cm = (

    np.sqrt(
        load_kn /
        (np.pi * pressure_kpa)
    )
    * 100.0

)


print()
print("Row 23 structure")
print("----------------")

print(
    f"Thickness = {thickness_cm} cm"
)

print(
    f"Poisson   = {poisson}"
)

print(
    f"Pressure  = {pressure_kpa} kPa"
)

print(
    f"Load      = {load_kn} kN"
)

print(
    f"Radius    = {contact_radius_cm:.4f} cm"
)

print(
    f"Locations = {locations_cm} cm"
)


# ============================================================
# 4. ACTUAL ERAPAVE CALIBRATION DATA
# ============================================================
#
# These are ACTUAL ERAPAVE runs.
#
# Columns:
#
# E1 E2 E3 E4 D0 D30
#
# ------------------------------------------------------------
# IMPORTANT:
#
# The 450/200/130/20 case is intentionally NOT included yet.
#
# Run it in ERAPAVE first.
#
# Then enter its actual D0 and D30 below.
# ============================================================


# ============================================================
# ACTUAL ERAPAVE CALIBRATION DATA
# ============================================================
# Columns:
# E1, E2, E3, E4, D0, D30
#
# D0  = ERAPave displacement at x = 0 cm
# D30 = ERAPave displacement at x = 30 cm
# ============================================================

data = np.array([
    [840, 195,  95, 20, 584.6, 359.5],
    [700, 160,  90, 20, 653.5, 383.6],
    [650, 180,  90, 20, 645.6, 375.3],
    [580, 180,  90, 20, 666.3, 377.2],
    [550, 200,  90, 20, 657.5, 370.2],
    [500, 220, 120, 20, 628.4, 337.1],
    [450, 180, 120, 20, 685.3, 333.4],
    [450, 250, 160, 20, 601.0, 306.1],
    [450, 180, 140, 20, 669.4, 337.7],
    [625, 100,  50, 25, 816.3, 466.8],
    [600, 120,  50, 20, 824.0, 487.8],
])


# ============================================================
# 5. OPTIONAL NEW CALIBRATION POINT
# ============================================================
#
# AFTER you run:
#
# E1 = 450
# E2 = 200
# E3 = 130
# E4 = 20
#
# enter the actual ERAPAVE values here.
#
# Example:
#
# NEW_D0  = 685.3
# NEW_D30 = 333.4
#
# ONLY use these numbers if they are actually produced
# by ERAPAVE for this exact modulus combination.
# ============================================================


ADD_NEW_POINT = False

NEW_E = np.array([
    625.0,
    100.0,
    50.0,
    25.0
])

NEW_D0 = 816.30
NEW_D30 = 466.80



if ADD_NEW_POINT:

    if NEW_D0 is None or NEW_D30 is None:

        raise ValueError(
            "\nADD_NEW_POINT=True but NEW_D0 or NEW_D30 "
            "is still None.\n\n"
            "Run the case in ERAPAVE first and enter the "
            "actual x=0 and x=30 displacement values."
        )


    new_row = np.array(

        [

            NEW_E[0],
            NEW_E[1],
            NEW_E[2],
            NEW_E[3],

            NEW_D0,
            NEW_D30

        ],

        dtype=float

    )


    data = np.vstack(
        [
            data,
            new_row
        ]
    )


    print()
    print(
        "NEW ERAPAVE CALIBRATION POINT ADDED:"
    )

    print(
        f"E = {NEW_E}"
    )

    print(
        f"D0  = {NEW_D0:.2f} um"
    )

    print(
        f"D30 = {NEW_D30:.2f} um"
    )


else:

    print()
    print(
        "No new calibration point added yet."
    )

    print(
        "Run 450/200/130/20 in ERAPAVE first."
    )


print()
print(
    f"Number of calibration runs = {len(data)}"
)


# ============================================================
# 6. RAW MLET FORWARD MODEL
# ============================================================

def raw_displacements(E):

    E = np.asarray(
        E,
        dtype=float
    )


    raw = raw_mlet.erapave_mlet(

        q_kpa=pressure_kpa,

        radius_cm=contact_radius_cm,

        r_cm=locations_cm,

        z_cm=np.zeros_like(
            locations_cm
        ),

        thickness_cm=thickness_cm,

        modulus_mpa=E,

        poisson=poisson,

        iteration=80,

        bounded=True

    )


    # --------------------------------------------------------
    # x = 0 cm
    # --------------------------------------------------------

    d0 = float(

        np.asarray(
            raw[0]
        ).ravel()[0]

    )


    # --------------------------------------------------------
    # x = 30 cm
    # --------------------------------------------------------

    d30 = float(

        np.asarray(
            raw[2]
        ).ravel()[0]

    )


    return d0, d30, raw


# ============================================================
# 7. BUILD RAW DATASET
# ============================================================

dataset = []


for row in data:

    E = row[:4]

    erapave_d0 = row[4]

    erapave_d30 = row[5]


    raw_d0, raw_d30, raw_all = raw_displacements(
        E
    )


    dataset.append(

        [

            E[0],
            E[1],
            E[2],
            E[3],

            raw_d0,
            raw_d30,

            erapave_d0,
            erapave_d30

        ]

    )


dataset = np.array(
    dataset,
    dtype=float
)


# ============================================================
# 8. REFERENCE MODULI
# ============================================================

E_REF = np.array(

    [
        600.0,
        200.0,
        110.0,
        20.0
    ],

    dtype=float
)


# ============================================================
# 9. FEATURE FUNCTION
# ============================================================

def make_features(E_values):

    E_values = np.asarray(
        E_values,
        dtype=float
    )


    if E_values.ndim == 1:

        E_values = E_values.reshape(
            1,
            -1
        )


    # Log-normalized modulus features

    X_modulus = np.log(
        E_values / E_REF
    )


    # Intercept

    X = np.column_stack(

        [

            np.ones(
                len(E_values)
            ),

            X_modulus

        ]

    )


    return X


# ============================================================
# 10. V2.2 FIT FUNCTION
# ============================================================

def fit_v22(

    E_values,

    raw_d0,
    raw_d30,

    target_d0,
    target_d30,

    ridge=0.05

):

    X = make_features(
        E_values
    )


    # Initial parameters

    p0 = np.array(

        [

            0.575,
            0.0,
            0.0,
            0.0,
            0.0,

            0.495,
            0.0,
            0.0,
            0.0,
            0.0

        ],

        dtype=float

    )


    def residual(p):

        # D0 coefficients

        b_local = p[:5]


        # D30 coefficients

        c_local = p[5:]


        # Correction factors

        factor0 = X @ b_local

        factor30 = X @ c_local


        # Predictions

        pred0 = (
            factor0
            * raw_d0
        )


        pred30 = (
            factor30
            * raw_d30
        )


        # Relative residuals

        r0 = (

            pred0
            - target_d0

        ) / target_d0


        r30 = (

            pred30
            - target_d30

        ) / target_d30


        # Ridge regularization

        penalty = (

            np.sqrt(ridge)
            *
            np.concatenate(

                [

                    b_local[1:],
                    c_local[1:]

                ]

            )

        )


        return np.concatenate(

            [

                r0,
                r30,
                penalty

            ]

        )


    result = least_squares(

        residual,

        p0

    )


    return result


# ============================================================
# 11. FIT FULL MODEL
# ============================================================

fit = fit_v22(

    dataset[:, :4],

    dataset[:, 4],
    dataset[:, 5],

    dataset[:, 6],
    dataset[:, 7],

    ridge=0.05

)


parameters = fit.x


b = parameters[:5]

c = parameters[5:]


# ============================================================
# 12. PRINT PARAMETERS
# ============================================================

print()
print("=" * 80)
print("V2.2 CALIBRATION PARAMETERS")
print("=" * 80)

print()

print(
    "D0 coefficients:"
)

print(
    b
)

print()

print(
    "D30 coefficients:"
)

print(
    c
)


# ============================================================
# 13. PREDICTION FUNCTION
# ============================================================

def predict_v22(

    E,

    raw_d0,
    raw_d30

):

    X = make_features(
        E
    )


    # IMPORTANT:
    # Convert one-element arrays safely to scalar.

    factor0 = float(

        (
            X @ b
        ).ravel()[0]

    )


    factor30 = float(

        (
            X @ c
        ).ravel()[0]

    )


    d0 = (
        factor0
        * raw_d0
    )


    d30 = (
        factor30
        * raw_d30
    )


    sci300 = (
        d0
        - d30
    )


    return (

        d0,
        d30,
        sci300,
        factor0,
        factor30

    )


# ============================================================
# 14. TRAINING PREDICTIONS
# ============================================================

predicted_d0 = []

predicted_d30 = []


for row in dataset:

    E = row[:4]

    raw0 = row[4]

    raw30 = row[5]


    p0, p30, _, _, _ = predict_v22(

        E,

        raw0,
        raw30

    )


    predicted_d0.append(
        p0
    )

    predicted_d30.append(
        p30
    )


predicted_d0 = np.array(
    predicted_d0
)

predicted_d30 = np.array(
    predicted_d30
)


# ============================================================
# 15. TRAINING ERRORS
# ============================================================

target_d0 = dataset[:, 6]

target_d30 = dataset[:, 7]


target_sci = (

    target_d0
    - target_d30

)


predicted_sci = (

    predicted_d0
    - predicted_d30

)


error_d0 = (

    100
    *
    (
        predicted_d0
        - target_d0
    )
    /
    target_d0

)


error_d30 = (

    100
    *
    (
        predicted_d30
        - target_d30
    )
    /
    target_d30

)


error_sci = (

    100
    *
    (
        predicted_sci
        - target_sci
    )
    /
    target_sci

)


# ============================================================
# 16. TRAINING MAPE
# ============================================================

training_mape_d0 = np.mean(
    np.abs(error_d0)
)

training_mape_d30 = np.mean(
    np.abs(error_d30)
)

training_mape_sci = np.mean(
    np.abs(error_sci)
)


print()
print("=" * 80)
print("V2.2 TRAINING PERFORMANCE")
print("=" * 80)

print(
    f"D0     MAPE = {training_mape_d0:.2f}%"
)

print(
    f"D30    MAPE = {training_mape_d30:.2f}%"
)

print(
    f"SCI300 MAPE = {training_mape_sci:.2f}%"
)


# ============================================================
# 17. TRAINING TABLE
# ============================================================

print()
print("=" * 120)

print(
    "V2.2 TRAINING PREDICTIONS"
)

print("=" * 120)


print(

    f"{'Run':>4}"
    f"{'E1':>7}"
    f"{'E2':>7}"
    f"{'E3':>7}"
    f"{'E4':>7}"
    f"{'ERA D0':>11}"
    f"{'Pred D0':>11}"
    f"{'ERA D30':>11}"
    f"{'Pred D30':>11}"
    f"{'D0 err%':>10}"
    f"{'D30 err%':>11}"
    f"{'SCI err%':>11}"

)

print(
    "-" * 120
)


for i, row in enumerate(dataset):

    print(

        f"{i:4d}"
        f"{row[0]:7.0f}"
        f"{row[1]:7.0f}"
        f"{row[2]:7.0f}"
        f"{row[3]:7.0f}"
        f"{row[6]:11.1f}"
        f"{predicted_d0[i]:11.1f}"
        f"{row[7]:11.1f}"
        f"{predicted_d30[i]:11.1f}"
        f"{error_d0[i]:10.2f}"
        f"{error_d30[i]:11.2f}"
        f"{error_sci[i]:11.2f}"

    )


# ============================================================
# 18. LEAVE-ONE-OUT VALIDATION
# ============================================================

print()
print("=" * 120)

print(
    "V2.2 LEAVE-ONE-OUT VALIDATION"
)

print("=" * 120)


print(

    f"{'Run':>4}"
    f"{'E1':>7}"
    f"{'E2':>7}"
    f"{'E3':>7}"
    f"{'E4':>7}"
    f"{'ERA D0':>11}"
    f"{'Pred D0':>11}"
    f"{'ERA D30':>11}"
    f"{'Pred D30':>11}"
    f"{'D0 err%':>10}"
    f"{'D30 err%':>11}"
    f"{'SCI err%':>11}"

)

print(
    "-" * 120
)


loo_results = []


for i in range(
    len(dataset)
):

    # Remove current observation

    train = np.delete(

        dataset,

        i,

        axis=0

    )


    # Fit remaining observations

    loo_fit = fit_v22(

        train[:, :4],

        train[:, 4],
        train[:, 5],

        train[:, 6],
        train[:, 7],

        ridge=0.05

    )


    p = loo_fit.x


    b_loo = p[:5]

    c_loo = p[5:]


    # Test observation

    X_test = make_features(

        dataset[i, :4]

    )


    # Safe scalar conversion

    factor0 = float(

        (
            X_test @ b_loo
        ).ravel()[0]

    )


    factor30 = float(

        (
            X_test @ c_loo
        ).ravel()[0]

    )


    # Predictions

    pred0 = (

        factor0
        * dataset[i, 4]

    )


    pred30 = (

        factor30
        * dataset[i, 5]

    )


    # ERAPAVE

    era0 = dataset[i, 6]

    era30 = dataset[i, 7]


    # SCI

    era_sci = (
        era0
        - era30
    )


    pred_sci = (
        pred0
        - pred30
    )


    # Errors

    e0 = (

        100
        *
        (
            pred0
            - era0
        )
        /
        era0

    )


    e30 = (

        100
        *
        (
            pred30
            - era30
        )
        /
        era30

    )


    esci = (

        100
        *
        (
            pred_sci
            - era_sci
        )
        /
        era_sci

    )


    loo_results.append(

        [

            i,

            *dataset[i, :4],

            era0,
            pred0,

            era30,
            pred30,

            era_sci,
            pred_sci,

            e0,
            e30,
            esci

        ]

    )


    print(

        f"{i:4d}"
        f"{dataset[i,0]:7.0f}"
        f"{dataset[i,1]:7.0f}"
        f"{dataset[i,2]:7.0f}"
        f"{dataset[i,3]:7.0f}"
        f"{era0:11.1f}"
        f"{pred0:11.1f}"
        f"{era30:11.1f}"
        f"{pred30:11.1f}"
        f"{e0:10.2f}"
        f"{e30:11.2f}"
        f"{esci:11.2f}"

    )


loo_results = np.array(
    loo_results
)


# ============================================================
# 19. LOO MAPE
# ============================================================

loo_d0_error = loo_results[:, 11]

loo_d30_error = loo_results[:, 12]

loo_sci_error = loo_results[:, 13]


loo_mape_d0 = np.mean(
    np.abs(loo_d0_error)
)

loo_mape_d30 = np.mean(
    np.abs(loo_d30_error)
)

loo_mape_sci = np.mean(
    np.abs(loo_sci_error)
)


print()
print("=" * 80)

print(
    "V2.2 LEAVE-ONE-OUT MAPE"
)

print("=" * 80)


print(
    f"D0     = {loo_mape_d0:.2f}%"
)

print(
    f"D30    = {loo_mape_d30:.2f}%"
)

print(
    f"SCI300 = {loo_mape_sci:.2f}%"
)


# ============================================================
# CORRECT TEST CASE: 450 / 200 / 130 / 20
# ============================================================

test_E = np.array([
    450.0,
    200.0,
    130.0,
    20.0
])


test_raw_d0, test_raw_d30, test_raw = raw_displacements(
    test_E
)


(
    test_d0,
    test_d30,
    test_sci,
    test_f0,
    test_f30
) = predict_v22(
    test_E,
    test_raw_d0,
    test_raw_d30
)


# ------------------------------------------------------------
# ACTUAL ERAPAVE RESULT FOR 450/200/130/20
# ------------------------------------------------------------

actual_d0 = 657.70

actual_d30 = 337.20

actual_sci = (
    actual_d0 -
    actual_d30
)


# ------------------------------------------------------------
# TEST ERRORS
# ------------------------------------------------------------

err_d0 = (
    100 *
    (test_d0 - actual_d0)
    / actual_d0
)


err_d30 = (
    100 *
    (test_d30 - actual_d30)
    / actual_d30
)


err_sci = (
    100 *
    (test_sci - actual_sci)
    / actual_sci
)


print()
print("=" * 80)
print("CORRECTED INDEPENDENT TEST: 450/200/130/20")
print("=" * 80)

print()

print(
    f"Actual D0     = {actual_d0:.2f} um"
)

print(
    f"Predicted D0  = {test_d0:.2f} um"
)

print(
    f"Error         = {err_d0:+.2f}%"
)

print()

print(
    f"Actual D30    = {actual_d30:.2f} um"
)

print(
    f"Predicted D30 = {test_d30:.2f} um"
)

print(
    f"Error         = {err_d30:+.2f}%"
)

print()

print(
    f"Actual SCI300    = {actual_sci:.2f} um"
)

print(
    f"Predicted SCI300 = {test_sci:.2f} um"
)

print(
    f"Error            = {err_sci:+.2f}%"
)

# ============================================================
# 21. IF THIS CASE HAS ACTUAL ERAPAVE RESULTS,
#     COMPARE THEM
# ============================================================

if NEW_D0 is not None and NEW_D30 is not None:

    actual_d0 = NEW_D0

    actual_d30 = NEW_D30

    actual_sci = (
        actual_d0
        - actual_d30
    )


    err_d0 = (

        100
        * (
            test_d0
            - actual_d0
        )
        /
        actual_d0

    )


    err_d30 = (

        100
        * (
            test_d30
            - actual_d30
        )
        /
        actual_d30

    )


    err_sci = (

        100
        * (
            test_sci
            - actual_sci
        )
        /
        actual_sci

    )


    print()
    print(
        "ACTUAL ERAPAVE RESULT"
    )

    print(
        f"D0     = {actual_d0:.2f} um"
    )

    print(
        f"D30    = {actual_d30:.2f} um"
    )

    print(
        f"SCI300 = {actual_sci:.2f} um"
    )


    print()
    print(
        "TEST ERRORS"
    )

    print(
        f"D0     = {err_d0:+.2f}%"
    )

    print(
        f"D30    = {err_d30:+.2f}%"
    )

    print(
        f"SCI300 = {err_sci:+.2f}%"
    )


# ============================================================
# 22. FINAL DECISION
# ============================================================

print()
print("=" * 80)

print(
    "V2.2 DECISION"
)

print("=" * 80)


if (

    loo_mape_d0 <= 2.0

    and

    loo_mape_d30 <= 2.0

    and

    loo_mape_sci <= 2.0

):

    print()
    print(
        "PROMISING."
    )

    print(
        "All three LOO MAPE values are below 2%."
    )

    print()
    print(
        "However, more independent pavement structures "
        "are still required before applying the model "
        "to the full 15,000-row dataset."
    )


else:

    print()
    print(
        "NOT READY for 15,000-row prediction."
    )

    print()
    print(
        "Add more ACTUAL ERAPAVE calibration runs."
    )

    print()
    print(
        "Priority region:"
    )

    print(
        "E1 = 450-550 MPa"
    )

    print(
        "E2 = 180-220 MPa"
    )

    print(
        "E3 = 120-160 MPa"
    )

    print(
        "E4 = 20 MPa"
    )


print()
print("=" * 80)

print(
    "END OF V2.2"
)

print("=" * 80)

Raw MLET solver loaded successfully.

Row 23 structure
----------------
Thickness = [13.989 37.085 88.263] cm
Poisson   = [0.35 0.35 0.35 0.4 ]
Pressure  = 700.0 kPa
Load      = 50.0 kN
Radius    = 15.0786 cm
Locations = [  0.  20.  30.  60.  90. 120. 150.] cm

No new calibration point added yet.
Run 450/200/130/20 in ERAPAVE first.

Number of calibration runs = 11

V2.2 CALIBRATION PARAMETERS

D0 coefficients:
[ 0.57509697 -0.04353688  0.00577711  0.01736555  0.01190458]

D30 coefficients:
[ 0.49678264  0.01370504  0.01876137 -0.01265244  0.01898864]

V2.2 TRAINING PERFORMANCE
D0     MAPE = 0.12%
D30    MAPE = 0.87%
SCI300 MAPE = 0.97%

V2.2 TRAINING PREDICTIONS
 Run     E1     E2     E3     E4     ERA D0    Pred D0    ERA D30   Pred D30   D0 err%   D30 err%   SCI err%
------------------------------------------------------------------------------------------------------------------------
   0    840    195     95     20      584.6      584.7      359.5      362.2      0.01       0.76 